## Resolución de la ecuación de Laplace:
con condiciones Dirichlet no homogéneas:
$$\begin{array}{rl}-\Delta u = 1 & \text{en $\Omega$,}\\
u = x+2y & \text{en $\Gamma_1$,} \\
u = 3 & \text{en $\Gamma_2$,}
\end{array}$$
con $\Omega = D((0,0),1)\backslash D((\frac35,0),\frac13)$, $\Gamma_1 = \partial D((0,0),1)$, $\Gamma_2 = \partial D((\frac35,0), \frac13)$

Etiquetas: $\Gamma_1$: 1; $\Gamma_2$: 2

### Importamos módulos

In [21]:
import mfem.ser as mfem
from glvis import glvis

### Lectura de malla

La malla ha sido creada con FreeFem, exportada a formato nativo (`.msh`), y posteriormente convertida a formato `.mesh` usando `convert2mfem.py`

In [22]:
mesh = mfem.Mesh('mallas/doblecirc.mesh')
glvis(mesh)

#### Información básica de la malla

In [23]:
print("Número de elementos:", mesh.GetNE())
print("Número de vértices:",mesh.GetNV())
dim = mesh.Dimension()

Número de elementos: 692
Número de vértices: 386


### Espacio de elementos finitos

In [24]:
order = 1
fec = mfem.H1_FECollection(order, dim)
fespace = mfem.FiniteElementSpace(mesh, fec)
print('Número de incógnitas:',fespace.GetTrueVSize())

Número de incógnitas: 386


### Formulación variacional
$$\text{Hallar }u\in u_0 + H^1_0(\Omega):\quad \int_\Omega \nabla u \cdot \nabla v\,dx = \int_\Omega 1\cdot v\,dx,\quad \forall v \in H^1_0(\Omega)$$

In [25]:
# forma bilineal
a = mfem.BilinearForm(fespace)
a.AddDomainIntegrator(mfem.DiffusionIntegrator())
a.Assemble()

# Segundo miembro (f=1)
one = mfem.ConstantCoefficient(1.0)

b = mfem.LinearForm(fespace)
b.AddDomainIntegrator(mfem.DomainLFIntegrator(one))
b.Assemble()

### Condiciones en la frontera

Como todas las fronteras son Dirichlet:

In [26]:
boundary_dofs = mfem.intArray()
# Obtenemos etiquetas del espacio de elementos finitos
fespace.GetBoundaryTrueDofs(boundary_dofs)

#### Definimos vector de valores en la frontera (mediante `GridFunction`)

##### Etiquetas de la frontera
Para marcar las etiquetas en la frontera, MFEM usa un array de ceros y unos de longitud igual al mayor valor existente de las etiquetas. La forma de imponer condiciones sobre una parte de la frontera será poniendo un 1 en la posición de la máscara que corresponda a la etiqueta que queramos marcar.

En el ejemplo, las etiquetas de la malla en FreeFem++ han sido puestas a 1 ($\Gamma_1$) y 2 ($\Gamma_2$). Entonces el atributo 
<code>
mesh.bdr_attributes.ToList()
</code>
proporciona
<code>
[1,2]
</code>

In [27]:
mesh.bdr_attributes.ToList()

[1, 2]

In [33]:
x = mfem.GridFunction(fespace)
x.Assign(0.)
inner = [0,1]
inner_border = mfem.intArray(inner)

coef = mfem.ConstantCoefficient(3.)
x.ProjectBdrCoefficient(coef,inner_border)


In [34]:
class Outer_Fun(mfem.PyCoefficient):
    def EvalValue(self,x):
        return x[0] + 2*x[1]
u0 = Outer_Fun()

outer = [1,0]
outer_border = mfem.intArray(outer)
x.ProjectBdrCoefficient(u0,outer_border)

### Formulación del sistema

In [30]:
A = mfem.SparseMatrix()
B = mfem.Vector()
X = mfem.Vector()

a.FormLinearSystem(boundary_dofs, x, b, A, X, B)

###  Resolución del sistema

In [31]:
mfem.CG(A, B, X, 0, 200, 1e-12, 0.0)

# Asignamos solución a la función grid
a.RecoverFEMSolution(X, b, x)

### Visualización PYGLVIS

In [35]:
glvis((mesh, x),keys="Rcppp") 